In [3]:
df = pd.read_csv("C:/Users/jothi/Downloads/Fake review dataset/fake reviews dataset.csv")

In [4]:
print(df.shape)

(40432, 4)


In [5]:
df.describe()

,rating
count,40432.000000
mean,4.256579
std,1.144354
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [6]:

df['category'].value_counts()


category
Kindle_Store_5                  4730
Books_5                         4370
Pet_Supplies_5                  4254
Home_and_Kitchen_5              4056
Electronics_5                   3988
Sports_and_Outdoors_5           3946
Tools_and_Home_Improvement_5    3858
Clothing_Shoes_and_Jewelry_5    3848
Toys_and_Games_5                3794
Movies_and_TV_5                 3588
Name: count, dtype: int64

In [7]:
df['label'].value_counts()

label
CG    20216
OR    20216
Name: count, dtype: int64

In [8]:
import sqlite3

conn = sqlite3.connect("reviews.db")
df.to_sql("reviews", conn, if_exists="replace", index=True, index_label="review_id")
conn.commit()

In [9]:

pd.read_sql("select label , count(*) ,avg(rating) from reviews group by label ;",conn)


,label,count(*),avg(rating)
0,CG,20216,4.259893
1,OR,20216,4.253265


In [10]:
pd.read_sql("select label,rating,count(rating) from reviews group by label,rating;",
            conn)

,label,rating,count(rating)
0,CG,1.0,1063
1,CG,2.0,962
2,CG,3.0,1952
3,CG,4.0,3920
4,CG,5.0,12319
5,OR,1.0,1092
6,OR,2.0,1005
7,OR,3.0,1834
8,OR,4.0,4045
9,OR,5.0,12240


In [11]:
print(df.columns)

Index(['category', 'rating', 'label', 'text_'], dtype='object')


In [12]:
df["text_"].str.len()

0          75
1          80
2          67
3          81
4          85
         ... 
40427    1694
40428    1304
40429    1987
40430    1301
40431    1768
Name: text_, Length: 40432, dtype: int64

In [13]:
import pandas as pd

In [14]:
import nltk
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

df['sentiment'] = df['text_'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

df.groupby('label')['sentiment'].agg(['mean', 'std'])

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\jothi\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


,mean,std
label,,
CG,0.634677,0.449289
OR,0.582335,0.488984


In [15]:
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')

from nltk import pos_tag, word_tokenize
import pandas as pd

def pos_ratios(text):
    tokens = word_tokenize(str(text))
    tags = pos_tag(tokens)
    n = len(tags) if tags else 1
    adj = sum(1 for _, t in tags if t.startswith('JJ'))
    noun = sum(1 for _, t in tags if t.startswith('NN'))
    return adj/n, noun/n

sample = df.sample(5000, random_state=42).copy()
sample[['adj_ratio','noun_ratio']] = sample['text_'].apply(lambda x: pd.Series(pos_ratios(x)))
sample.groupby('label')[['adj_ratio','noun_ratio']].mean()

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\jothi\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jothi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,adj_ratio,noun_ratio
label,,
CG,0.094792,0.180196
OR,0.084824,0.204493


In [16]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\jothi\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [17]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jothi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [18]:
def ttr(text):
    tokens = str(text).lower().split()
    return len(set(tokens)) / len(tokens) if tokens else 0

df['lex_diversity'] = df['text_'].apply(ttr)
df.groupby('label')['lex_diversity'].mean()

label
CG    0.750173
OR    0.825091
Name: lex_diversity, dtype: float64

In [19]:
df['text_len'] = df['text_'].str.len()
df.groupby('label')['text_len'].mean()

label
CG    305.573506
OR    396.970419
Name: text_len, dtype: float64

In [20]:
generic_phrases = ['highly recommend', 'great product', 'five stars', 'works great',
                    'love it', 'would recommend', 'great price', 'easy to use']

def generic_count(text):
    text = str(text).lower()
    return sum(text.count(p) for p in generic_phrases)

df['generic_phrase_count'] = df['text_'].apply(generic_count)
df.groupby('label')['generic_phrase_count'].mean()

label
CG    0.347942
OR    0.106500
Name: generic_phrase_count, dtype: float64

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Combine our 5 features
features = df[['sentiment', 'lex_diversity', 'generic_phrase_count', 'text_len']].copy()

# Add noun_ratio and adj_ratio from the 'sample' dataframe — but those were only on 5000 rows
# For now we'll build the model on the full data using the 4 features we calculated on all 40k rows

target = df['label'].map({'CG': 1, 'OR': 0})  # 1 = fake, 0 = real

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.73      0.76      0.74      4071
           1       0.74      0.71      0.73      4016

    accuracy                           0.74      8087
   macro avg       0.74      0.74      0.74      8087
weighted avg       0.74      0.74      0.74      8087



In [22]:
import pandas as pd

coefficients = pd.Series(model.coef_[0], index=features.columns)
coefficients.sort_values(ascending=False)

generic_phrase_count     1.023109
sentiment                0.028614
text_len                -0.006190
lex_diversity          -13.262116
dtype: float64

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

results = {}

for cat in df['category'].unique():
    sub = df[df['category'] == cat]
    X = sub[['sentiment', 'lex_diversity', 'generic_phrase_count', 'text_len']]
    y = sub['label'].map({'CG': 1, 'OR': 0})
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    m = LogisticRegression()
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    
    results[cat] = acc

pd.Series(results).sort_values(ascending=False)

C:\Users\jothi\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Movies_and_TV_5                 0.805014
Books_5                         0.774600
Kindle_Store_5                  0.770613
Electronics_5                   0.741855
Toys_and_Games_5                0.741765
Sports_and_Outdoors_5           0.730380
Tools_and_Home_Improvement_5    0.718912
Home_and_Kitchen_5              0.714286
Pet_Supplies_5                  0.708578
Clothing_Shoes_and_Jewelry_5    0.681818
dtype: float64

In [24]:
# Save our engineered features into the SQLite database you already created
df.to_sql("reviews_features", conn, if_exists="replace", index=True, index_label="review_id")

# Query 1: category vulnerability ranking (same numbers as above, but from SQL this time)
pd.read_sql("""
    SELECT category, label, COUNT(*) as cnt, AVG(generic_phrase_count) as avg_generic, AVG(lex_diversity) as avg_diversity
    FROM reviews_features
    GROUP BY category, label
    ORDER BY category
""", conn)

,category,label,cnt,avg_generic,avg_diversity
0,Books_5,CG,2185,0.295652,0.695838
1,Books_5,OR,2185,0.066362,0.802865
2,Clothing_Shoes_and_Jewelry_5,CG,1924,0.150208,0.792118
3,Clothing_Shoes_and_Jewelry_5,OR,1924,0.085759,0.844282
4,Electronics_5,CG,1994,0.552658,0.748346
5,Electronics_5,OR,1994,0.153962,0.818555
6,Home_and_Kitchen_5,CG,2028,0.390533,0.775422
7,Home_and_Kitchen_5,OR,2028,0.140039,0.836376
8,Kindle_Store_5,CG,2365,0.309091,0.707195
9,Kindle_Store_5,OR,2365,0.073150,0.793629


In [25]:
# Query 2: overall fraud risk ranking by category (using generic phrase gap as proxy)
pd.read_sql("""
    SELECT category,
           AVG(CASE WHEN label='CG' THEN generic_phrase_count END) as cg_avg,
           AVG(CASE WHEN label='OR' THEN generic_phrase_count END) as or_avg,
           AVG(CASE WHEN label='CG' THEN generic_phrase_count END) - 
           AVG(CASE WHEN label='OR' THEN generic_phrase_count END) as gap
    FROM reviews_features
    GROUP BY category
    ORDER BY gap DESC
""", conn)

,category,cg_avg,or_avg,gap
0,Electronics_5,0.552658,0.153962,0.398696
1,Tools_and_Home_Improvement_5,0.433904,0.138932,0.294971
2,Toys_and_Games_5,0.374802,0.102794,0.272008
3,Home_and_Kitchen_5,0.390533,0.140039,0.250493
4,Pet_Supplies_5,0.372355,0.131641,0.240715
5,Kindle_Store_5,0.309091,0.073150,0.235941
6,Sports_and_Outdoors_5,0.354283,0.119615,0.234668
7,Books_5,0.295652,0.066362,0.229291
8,Movies_and_TV_5,0.242475,0.055741,0.186734
9,Clothing_Shoes_and_Jewelry_5,0.150208,0.085759,0.064449


In [26]:
df.to_csv("reviews_dashboard_export.csv", index=True, index_label="review_id")

In [27]:
from nltk import pos_tag, word_tokenize

def pos_ratios(text):
    tokens = word_tokenize(str(text))
    tags = pos_tag(tokens)
    n = len(tags) if tags else 1
    adj = sum(1 for _, t in tags if t.startswith('JJ'))
    noun = sum(1 for _, t in tags if t.startswith('NN'))
    return adj/n, noun/n

df[['adj_ratio','noun_ratio']] = df['text_'].apply(lambda x: pd.Series(pos_ratios(x)))
df.groupby('label')[['adj_ratio','noun_ratio']].mean()

,adj_ratio,noun_ratio
label,,
CG,0.093567,0.180001
OR,0.084179,0.204806


In [28]:
df.to_csv("reviews_dashboard_export_full.csv", index=True, index_label="review_id")

In [29]:
from sklearn.preprocessing import MinMaxScaler

# Normalize each feature to 0-1 so they're comparable
features_to_combine = ['sentiment', 'lex_diversity', 'generic_phrase_count', 'adj_ratio', 'noun_ratio']
scaler = MinMaxScaler()
scaled = pd.DataFrame(scaler.fit_transform(df[features_to_combine]), columns=[f+'_scaled' for f in features_to_combine], index=df.index)

# Combine using the model's learned direction (lex_diversity and noun_ratio push toward REAL, others toward FAKE)
df['fraud_risk_score'] = (
    (1 - scaled['lex_diversity_scaled']) * 0.45 +
    scaled['generic_phrase_count_scaled'] * 0.30 +
    (1 - scaled['noun_ratio_scaled']) * 0.15 +
    scaled['adj_ratio_scaled'] * 0.05 +
    scaled['sentiment_scaled'] * 0.05
)

df[['label', 'fraud_risk_score']].groupby('label').mean()

,fraud_risk_score
label,
CG,0.296724
OR,0.251808


In [30]:
df['fraud_risk_score'] = model.predict_proba(df[['sentiment', 'lex_diversity', 'generic_phrase_count', 'text_len']])[:, 1]
df.groupby('label')['fraud_risk_score'].agg(['mean', 'std'])

,mean,std
label,,
CG,0.647308,0.238257
OR,0.355102,0.212005


In [31]:
df.sort_values('fraud_risk_score', ascending=False)[['category', 'rating', 'label', 'text_', 'fraud_risk_score']].head(10)

,category,rating,label,text_,fraud_risk_score
11856,Electronics_5,5.0,CG,I decided on this model because it was the bes...,1.000000
3900,Home_and_Kitchen_5,5.0,CG,Background\nThis was a replacement for a stand...,0.999998
11622,Electronics_5,5.0,CG,I've been a photographer for 13 years and have...,0.999985
32599,Books_5,4.0,CG,I have proudly been a member of the American L...,0.999977
4574,Sports_and_Outdoors_5,5.0,OR,Great product because it worked! Great product...,0.999970
39768,Clothing_Shoes_and_Jewelry_5,5.0,CG,I HATE WEARING BRAS! I HATE BRAS! I HATE BRAS!...,0.999951
15470,Movies_and_TV_5,5.0,CG,I just plain love the show. The characters are...,0.999950
31041,Books_5,5.0,CG,A good book club book. It's a good book club ...,0.999912
14261,Movies_and_TV_5,4.0,CG,Twisted Premise? Check\nDark? Check\n\nDark?...,0.999893
17882,Tools_and_Home_Improvement_5,5.0,CG,First step in home automation. The first step ...,0.999852


In [32]:
# Save fraud_risk_score into the database alongside everything else
df.to_sql("reviews_features", conn, if_exists="replace", index=True, index_label="review_id")

# Re-export the CSV too, now including fraud_risk_score
df.to_csv("reviews_dashboard_export_full.csv", index=True, index_label="review_id")